In [2]:
import torch
import time

### Device Check

In [3]:
torch.cuda

<module 'torch.cuda' from 'C:\\Users\\PRASHANTH N\\PycharmProjects\\MTechSem2\\gpu\\.venv\\Lib\\site-packages\\torch\\cuda\\__init__.py'>

In [4]:
torch.cuda.device

torch.cuda.device

In [5]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())

NVIDIA GeForce RTX 5050 Laptop GPU


In [6]:
if torch.cuda.is_available():
    print(torch.cuda.temperature())

42


In [7]:
if torch.cuda.is_available():
    try:
        print(torch.cuda.memory_usage())
    except Exception as e:
        print(e)

0


In [8]:
if torch.cuda.is_available():
    print(torch.cuda.get_allocator_backend())

native


In [9]:
torch.cuda.device_count()

1

In [10]:
print(torch.cuda.get_device_properties(0))

_CudaDeviceProperties(name='NVIDIA GeForce RTX 5050 Laptop GPU', major=12, minor=0, total_memory=8150MB, multi_processor_count=20, uuid=7bebb1ac-d67d-9d19-cf82-f9c782c819d5, pci_bus_id=4, pci_device_id=0, pci_domain_id=0, L2_cache_size=32MB)


In [11]:
print(torch.cuda.max_memory_allocated() / 1024**3, "GB")

0.0 GB


In [12]:
torch.cuda.is_available()

True

In [13]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
device

'cuda'

### Data on GPU

In [14]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [15]:
M.device

device(type='cpu')

In [16]:
M = M.to(device)
M

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

In [17]:
M.device

device(type='cuda', index=0)

In [18]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)
M.device

device(type='cuda', index=0)

In [19]:
M.device

device(type='cuda', index=0)

In [20]:
n = 10000
start = time.time()
M1_cpu = torch.rand(n, n)
end = time.time()
print("CPU Allocation Time : ", end-start)
start = time.time()
M2_gpu = torch.rand(n, n, device=device)
end = time.time()
print("GPU Allocation Time : ", end-start)

CPU Allocation Time :  0.3549001216888428
GPU Allocation Time :  0.01199960708618164


In [21]:
start = time.time()
M1_cpu @ M1_cpu.T
end = time.time()
print("CPU Run Time : ", end-start)

CPU Run Time :  5.953519105911255


In [22]:
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()
M2_gpu @ M2_gpu.T
end.record()
torch.cuda.synchronize()
print("GPU Run time:", start.elapsed_time(end) / 1000, "seconds")

GPU Run time: 0.3486094970703125 seconds


In [23]:
n = 10000
r = 10

torch.cuda.empty_cache()
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

A_gpu = torch.rand(n, n, device=device) # Float 32 dtype as default
print(A_gpu.dtype)

start.record()

for i in range(r): # A^r of A nxn size
    A_gpu = A_gpu @ A_gpu.T

end.record()
torch.cuda.synchronize()

print("GPU time:", start.elapsed_time(end) / 1000, "seconds")
print(torch.cuda.max_memory_allocated() / 1024**3, "GB")
"""
Profiling :
GPU Run time = 3 sec at n = 10000 (above 40k+ gives cuda out of mem) and A^x and x = 10
"""

torch.float32
GPU time: 3.033044677734375 seconds
1.1503915786743164 GB


'\nProfiling :\nGPU Run time = 3 sec at n = 10000 (above 40k+ gives cuda out of mem) and A^x and x = 10\n'